In [1]:
# ============================================
# STEP 1: Install required packages
# ============================================
!pip install pennylane pennylane-lightning -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 34.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 75.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 10.1 MB/s eta 0:00:00


In [2]:
# ============================================
# STEP 2: Import core libraries and log their versions
# ============================================
import torch
import torchvision
import numpy as np
import pennylane as qml
import sklearn
import matplotlib
import pandas as pd

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("NumPy:", np.__version__)
print("PennyLane:", qml.__version__)
print("scikit-learn:", sklearn.__version__)

PyTorch: 2.11.0+cpu
Torchvision: 0.26.0+cpu
NumPy: 2.0.2
PennyLane: 0.45.1
scikit-learn: 1.6.1


In [3]:
# ============================================
# STEP 3: Mount Google Drive
# ============================================
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
# ============================================
# STEP 4: Create the project folder structure
# ============================================
import os

base = "/content/drive/MyDrive/Quantum_DL_MNIST"

folders = [
    f"{base}/notebooks",
    f"{base}/data",
    f"{base}/splits",
    f"{base}/models",
    f"{base}/results/NN",
    f"{base}/results/CNN",
    f"{base}/results/CNN_Classical",
    f"{base}/results/CNN_Quantum",
    f"{base}/figures",
    f"{base}/documentation",
]

for f in folders:
    os.makedirs(f, exist_ok=True)
    print("Created:", f)

Created: /content/drive/MyDrive/Quantum_DL_MNIST/notebooks
Created: /content/drive/MyDrive/Quantum_DL_MNIST/data
Created: /content/drive/MyDrive/Quantum_DL_MNIST/splits
Created: /content/drive/MyDrive/Quantum_DL_MNIST/models
Created: /content/drive/MyDrive/Quantum_DL_MNIST/results/NN
Created: /content/drive/MyDrive/Quantum_DL_MNIST/results/CNN
Created: /content/drive/MyDrive/Quantum_DL_MNIST/results/CNN_Classical
Created: /content/drive/MyDrive/Quantum_DL_MNIST/results/CNN_Quantum
Created: /content/drive/MyDrive/Quantum_DL_MNIST/figures
Created: /content/drive/MyDrive/Quantum_DL_MNIST/documentation


In [5]:
# ============================================
# STEP 5: Download the OFFICIAL MNIST train and test sets
# ============================================
import torchvision
import torchvision.transforms as transforms

# Download official MNIST train and test sets separately (Section 2.1)
train_full = torchvision.datasets.MNIST(
    root=f"{base}/data",
    train=True,
    download=True,
    transform=transforms.ToTensor()
)

test_full = torchvision.datasets.MNIST(
    root=f"{base}/data",
    train=False,
    download=True,
    transform=transforms.ToTensor()
)

print("Full official train size:", len(train_full))
print("Full official test size:", len(test_full))

100%|██████████| 9.91M/9.91M [00:00<00:00, 36.0MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.21MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.48MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.4MB/s]


Full official train size: 60000
Full official test size: 10000


In [6]:
# ============================================================
# Step 6: Keep only digits 0 and 1 from both train and test sets
# ============================================================
import numpy as np

def filter_binary(dataset):
    """Keep only digits 0 and 1 from a full MNIST dataset."""
    # dataset.targets is a tensor of all labels (0-9)
    labels = dataset.targets
    mask = (labels == 0) | (labels == 1)
    indices = np.where(mask.numpy())[0]
    return indices

train_binary_idx = filter_binary(train_full)
test_binary_idx = filter_binary(test_full)

print("Binary train pool size:", len(train_binary_idx))
print("Binary test size:", len(test_binary_idx))

Binary train pool size: 12665
Binary test size: 2115


In [7]:
# ============================================================
# Step 7: Double-check class balance
# ============================================================
train_labels_binary = train_full.targets[train_binary_idx]
n_zeros = (train_labels_binary == 0).sum().item()
n_ones = (train_labels_binary == 1).sum().item()

print(f"Zeros: {n_zeros}  ({n_zeros/len(train_binary_idx)*100:.2f}%)")
print(f"Ones:  {n_ones}  ({n_ones/len(train_binary_idx)*100:.2f}%)")

Zeros: 5923  (46.77%)
Ones:  6742  (53.23%)


In [8]:
# ============================================================
# Step 7: Split the binary training pool into Train (80%) and
# Validation (20%), keeping the 0/1 class ratio balanced in both
# ============================================================

from sklearn.model_selection import train_test_split

# Labels for just the 0/1 training pool (needed for stratifying)
train_binary_labels = train_full.targets[train_binary_idx].numpy()

# Stratified split: keeps the same zero/one ratio in both Train and Val
train_idx, val_idx = train_test_split(
    train_binary_idx,
    test_size=0.20,
    stratify=train_binary_labels,
    random_state=42
)

print("Train size:", len(train_idx))
print("Val size:", len(val_idx))

Train size: 10132
Val size: 2533


In [9]:
# ============================================================
# Step 8: Save the train/val split indices to Drive
# So every other notebook (01-04) loads the SAME split
# ============================================================

splits_path = f"{base}/splits"

np.save(f"{splits_path}/train_indices.npy", train_idx)
np.save(f"{splits_path}/val_indices.npy", val_idx)
np.save(f"{splits_path}/test_indices.npy", test_binary_idx)

print("Saved train_indices.npy, val_indices.npy, test_indices.npy to:")
print(splits_path)

Saved train_indices.npy, val_indices.npy, test_indices.npy to:
/content/drive/MyDrive/Quantum_DL_MNIST/splits


In [10]:
# ============================================================
# Step 9: Compute normalization stats (mean & std) using ONLY
# the training split — never validation or test data
# ============================================================

# Pull out just the training images as a single tensor
train_images = train_full.data[train_idx].float() / 255.0  # scale to [0,1]

mean = train_images.mean().item()
std = train_images.std().item()

print("Training set mean:", mean)
print("Training set std:", std)

Training set mean: 0.12149777263402939
Training set std: 0.30103927850723267


In [11]:
# ============================================================
# Step 10: Save the normalization stats so all notebooks use
# the exact same values (computed once, reused everywhere)
# ============================================================

import json

norm_stats = {"mean": mean, "std": std}

with open(f"{splits_path}/normalization_stats.json", "w") as f:
    json.dump(norm_stats, f, indent=2)

print("Saved normalization_stats.json:")
print(norm_stats)

Saved normalization_stats.json:
{'mean': 0.12149777263402939, 'std': 0.30103927850723267}


In [12]:
# ============================================================
# Step 11: Build a custom Dataset class that applies our saved
# normalization stats to any subset of MNIST (train/val/test)
# ============================================================

from torch.utils.data import Dataset

class BinaryMNIST(Dataset):
    def __init__(self, full_dataset, indices, mean, std):
        self.full_dataset = full_dataset
        self.indices = indices
        self.mean = mean
        self.std = std

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        real_idx = self.indices[i]
        image, label = self.full_dataset[real_idx]  # image is [0,1] tensor, shape (1,28,28)
        image = (image - self.mean) / self.std       # apply normalization
        label = torch.tensor(float(label))            # 0.0 or 1.0, for BCEWithLogitsLoss
        return image, label

print("BinaryMNIST dataset class defined")

BinaryMNIST dataset class defined


In [13]:
# ============================================================
# Step 12: Create the actual Train, Val, and Test dataset objects
# using the saved indices and normalization stats
# ============================================================

import torch

train_dataset = BinaryMNIST(train_full, train_idx, mean, std)
val_dataset   = BinaryMNIST(train_full, val_idx, mean, std)
test_dataset  = BinaryMNIST(test_full, test_binary_idx, mean, std)

print("Train dataset size:", len(train_dataset))
print("Val dataset size:", len(val_dataset))
print("Test dataset size:", len(test_dataset))

# Quick sanity check: look at one sample
img, lbl = train_dataset[0]
print("Sample image shape:", img.shape, "| Sample label:", lbl.item())

Train dataset size: 10132
Val dataset size: 2533
Test dataset size: 2115
Sample image shape: torch.Size([1, 28, 28]) | Sample label: 1.0


In [14]:
# ============================================================
# Step 13: Wrap datasets in DataLoaders (handles batching + shuffling)
# num_workers=0 for reproducibility (fixed, non-negotiable setting)
# ============================================================

from torch.utils.data import DataLoader

BATCH_SIZE = 32

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print("DataLoaders ready")
print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

DataLoaders ready
Train batches: 317
Val batches: 80
Test batches: 67
